# M13 · ANN / vector search & indexing

_AFP-AI · Domain 2 · Retrieval & Representation_

**Trade a little exactness for the latency needed to search many vectors.**

We compare exact top-k search with a tiny approximate search. Quality is $\operatorname{recall@k}=\frac{|A_k\cap E_k|}{k}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

rng = np.random.default_rng(13)

## Synthetic creator embeddings

We make clustered vectors so approximate search can use a cheap coarse partition, similar in spirit to IVF.

In [ ]:
n_clusters = 8
items_per_cluster = 80
dim = 12
centers = rng.normal(size=(n_clusters, dim))
centers = centers / np.linalg.norm(centers, axis=1, keepdims=True)
labels = np.repeat(np.arange(n_clusters), items_per_cluster)
noise = rng.normal(scale=0.18, size=(n_clusters * items_per_cluster, dim))
items = centers[labels] + noise
items = items / np.linalg.norm(items, axis=1, keepdims=True)
query = centers[3] + centers[5] + rng.normal(scale=0.05, size=dim)
query = query / np.linalg.norm(query)

print(items.shape)

## Step 1 - Exact search

Exact search scores every vector and sorts the result. It is the quality reference.

In [ ]:
k = 10
start = time.perf_counter()
exact_scores = items @ query
exact_top = np.argsort(-exact_scores)[:k]
exact_time = time.perf_counter() - start

print("exact top ids:", exact_top)
print("exact time ms:", round(exact_time * 1000, 4))

assert len(exact_top) == k

## Step 2 - Approximate search with coarse clusters

We choose the nearest centroids, then score only items in those clusters. Increasing `nprobe` improves recall and costs more comparisons.

In [ ]:
def approx_search(nprobe):
    centroid_scores = centers @ query
    chosen_clusters = np.argsort(-centroid_scores)[:nprobe]
    mask = np.isin(labels, chosen_clusters)
    candidate_idx = np.where(mask)[0]
    candidate_scores = items[candidate_idx] @ query
    chosen_local = np.argsort(-candidate_scores)[:k]
    return candidate_idx[chosen_local], candidate_idx.size

approx_top, comparisons = approx_search(2)
overlap = len(set(exact_top).intersection(set(approx_top)))
recall = overlap / k

print("approx top ids:", approx_top)
print("comparisons:", comparisons)
print("recall@10:", recall)

assert comparisons < items.shape[0]

## Step 3 - Sweep the search knob

This is the recall-latency tradeoff in miniature.

In [ ]:
rows = []
for nprobe in range(1, n_clusters + 1):
    top_ids, count = approx_search(nprobe)
    hits = len(set(exact_top).intersection(set(top_ids)))
    rows.append({"nprobe": nprobe, "comparisons": count, "recall": hits / k})

sweep = pd.DataFrame(rows)

print(sweep)

assert sweep["recall"].iloc[-1] == 1.0

## Visualize recall vs comparisons

The curve shows why index tuning is a product decision, not only an algorithms decision.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sweep["comparisons"], sweep["recall"], marker="o")
ax.set_xlabel("vectors scored")
ax.set_ylabel("recall@10")
ax.set_ylim(0, 1.05)
ax.set_title("ANN recall tradeoff")
plt.show()

## Practice

Change `items_per_cluster`, `dim`, or `nprobe`. Try to find the smallest number of comparisons that still reaches recall@10 of at least 0.9.

In [ ]:
# Your turn:
